# Обучение модели обнаружения аномалий (Google Colab)

Ноутбук обучает **Isolation Forest** на штатном режиме (`scenario == 'normal'`) и сохраняет модель для последующей детекции. Это исправляет главную проблему исходного кода — модель обучалась на тех же данных, что и оценивала (data leakage) и нигде не сохранялась.

**Что получится в конце:** файлы `scaler.joblib` и `iforest.joblib`, которые подхватывает `anomaly_detection.detect_anomalies`.

Запускайте по одной ячейке сверху вниз. Локальный аналог без Colab — `notebooks/train_model_colab.py`.

## 1. Установка зависимостей

В Colab numpy/pandas обычно уже есть, но sklearn и joblib поставим явно.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib

## 2. Получение кода проекта и данных

Клонируем репозиторий с кодом `preprocessing.py`, `anomaly_detection.py`, `train_model.py`. Если репозиторий приватный — загрузите файлы вручную через панель слева (Files → Upload).

Данные: для оценки качества используем синтетику с разметкой сценариев. Для реальной тренировки подставьте свой CSV (`timestamp, sensor_id, temperature`).

In [ ]:
import os, sys, numpy as np, pandas as pd

REPO = "temperature-anomaly-monitor"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/zuwer-tech/temperature-anomaly-monitor.git
sys.path.insert(0, REPO)   # чтобы импортировать модули проекта
os.chdir(REPO)
print("Рабочая папка:", os.getcwd())

In [ ]:
# Синтетические данные с разметкой (компактный аналог Data.py).
def make_synth_df(n=300, num_sensors=6, seed=42):
    rng = np.random.RandomState(seed)
    t = np.arange(n)
    ts = pd.date_range("2026-06-06 12:00", periods=n, freq="1min")
    rows = []
    for s in range(1, num_sensors + 1):
        sid = f"T-{s:02d}"; shift = rng.uniform(-1.5, 1.5)
        temp = 70.0 + shift + 2*np.sin(2*np.pi*t/180) + rng.normal(0, 1.0, n)
        scen = np.array(["normal"]*n, dtype=object)
        if sid == "T-02":
            temp[120:140] += 15; scen[120:140] = "sharp_jump"
            temp[250:280] += np.linspace(0,8,30); scen[250:280] = "correlated_growth"
        if sid == "T-03":
            temp[160:240] += np.linspace(0,12,80); scen[160:240] = "slow_overheating"
        if sid == "T-04":
            temp[150:] += np.linspace(0,10,n-150); scen[150:] = "sensor_drift"
        if sid == "T-05":
            temp[180:220] = temp[180]; scen[180:220] = "stuck_sensor"
        if sid == "T-06":
            temp[80:120] += rng.normal(0,5,40); scen[80:120] = "high_noise"
            miss = np.random.choice(np.arange(80,120), size=8, replace=False)
            temp[miss] = np.nan; scen[miss] = "signal_loss"
        for i in range(n):
            rows.append({"timestamp": ts[i], "sensor_id": sid, "temperature": temp[i], "scenario": scen[i]})
    return pd.DataFrame(rows)

raw = make_synth_df()
print(raw.shape, "| сценарии:", raw["scenario"].unique().tolist())
raw.head()

## 3. Предобработка

`preprocess_data` считает признаки (rolling mean/std, z-score, is_stuck, отклонение от группы). На выходе DataFrame, готовый к обучению и детекции.

In [ ]:
from preprocessing import preprocess_data
df = preprocess_data(raw)
print("Предобработано:", df.shape)
df.head()

## 4. Обучение модели на штатном режиме

Ключевой момент: обучаемся **только на `scenario == 'normal'`**. Модель учит, как выглядит норма, а аномалии становятся для неё «нетипичными». `contamination` — ожидаемая доля аномалий (0.04 = 4%).

In [ ]:
import train_model
scaler, model, info = train_model.train(df, contamination=0.04)
print(f"Обучено на {info['train_rows']} строк normal | trained_on_normal={info['trained_on_normal']}")

## 5. Оценка качества

Так как у синтетики есть разметка `scenario`, считаем precision/recall/F1 и recall по каждому сценарию. Аномалия = `scenario != 'normal'`.

Для своих данных без разметки этот шаг покажет только число найденных аномалий.

In [ ]:
report = train_model.evaluate(df, scaler, model)
if "precision" in report:
    print(f"precision={report['precision']}  recall={report['recall']}  f1={report['f1']}")
    for scen, m in report["per_scenario"].items():
        print(f"  {scen:20} total={m['total']} detected={m['detected']} recall={m['recall']}")
else:
    print("Нет разметки scenario — качество не оценить.", report)

## 6. Сохранение и скачивание модели

Сохраняем `models/scaler.joblib`, `models/iforest.joblib`, `models/model_meta.json`. После этого `detect_anomalies` будет их загружать вместо обучения на лету.

Чтобы использовать модель в локальном проекте — скачайте три файла в папку `models/`.

In [ ]:
train_model.save_model(scaler, model, info, contamination=0.04)
print("Сохранено:", os.listdir("models"))

try:
    from google.colab import files
    for f in ["scaler.joblib", "iforest.joblib", "model_meta.json"]:
        files.download(os.path.join("models", f))
except Exception:
    print("Не в Colab — файлы лежат в models/.")